# Causal steering smoke run on Gemma 4 E4B — measured J-space interventions

Adds **measured** residual-stream interventions at the exact `block_output`
sites the frozen pilot lens reads (layers 35 and 38), using the completed
J-space run's k=10 cone records to define the deltas:

    h_intervened[row, pos] = h_original[row, pos] + multiplier * delta

Families: `output_atom_contribution`, `top_non_output_atom_contribution`,
`full_cone_reconstruction`, plus a deterministic, exactly norm-matched
`isotropic_random_direction` control per targeted condition. Multipliers
−1 / 0 / +1.

Hard boundaries, identical to previous runs:

- the pilot lens is **frozen** — fingerprint-verified, never refitted;
- the completed jspace run is **read-only** — cones are consumed, never rerun;
- model parameters are frozen; positions/deltas/outputs are finiteness-checked;
- a **baseline-parity gate** (unhooked vs multiplier-0 vs identical-copy
  writeback) must pass before any intervention is recorded;
- every condition has a deterministic ID; the run checkpoints after each
  condition and resumes append-safely; completed runs refuse resumption;
- causal claims are limited to what these records measure.

Safe end-to-end on an L4 (~120 conditions, ≈30–45 min including model load).
Run top to bottom in Colab; outside Colab every model-touching cell is a
gated no-op (the light path), so the structure can be validated locally.

## 0. Colab setup (skip if running locally)

In [ ]:
# 0. Colab bootstrap: clone/update the private repo from a fresh runtime.
# No-op outside Colab. Never loads Gemma; only touches git/pip.
import base64
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
    print("Not running in Colab — skipping bootstrap; using the local checkout.")
else:
    CHECKOUT_DIR = Path("/content/jacobian-lens-gemma")
    REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
    BRANCH = "multimodal-jlens-explorer"

    def _normalize(url: str) -> str:
        return url.strip().removesuffix(".git").removesuffix("/")

    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_TOKEN secret not found or not accessible. In Colab: open "
            "the key icon (Secrets) in the left sidebar, add a secret named "
            "GITHUB_TOKEN containing a fine-grained GitHub token with "
            "read-only access to MechInterpreter/jacobian-lens-gemma, and "
            "enable notebook access for it. Then re-run this cell."
        )

    # Auth via a per-invocation extraHeader override: lives only in argv,
    # never written to .git/config, the remote URL, or notebook output.
    _token_b64 = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
    _auth_header = f"AUTHORIZATION: basic {_token_b64}"
    _auth_args = ["-c", f"http.https://github.com/.extraHeader={_auth_header}"]

    def _run(args, *, auth=False, check=True):
        cmd = ["git", *(_auth_args if auth else []), *args]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if check and result.returncode != 0:
            safe_stderr = result.stderr.replace(GITHUB_TOKEN, "***")
            raise RuntimeError(f"git {' '.join(args)} failed:\n{safe_stderr}")
        return result.stdout.strip()

    if not CHECKOUT_DIR.exists():
        print(f"Cloning {REPO_URL} ({BRANCH}) into {CHECKOUT_DIR} ...")
        _run(["clone", "--branch", BRANCH, REPO_URL, str(CHECKOUT_DIR)], auth=True)
    else:
        if not (CHECKOUT_DIR / ".git").exists():
            raise RuntimeError(
                f"{CHECKOUT_DIR} exists but is not a git checkout; refusing to "
                "touch it. Remove or rename it manually, then re-run this cell."
            )
        existing_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
        if _normalize(existing_url) != _normalize(REPO_URL):
            raise RuntimeError(
                f"{CHECKOUT_DIR} is checked out from {existing_url!r}, not "
                f"{REPO_URL!r}; refusing to touch an unexpected repository. "
                "Remove or rename it manually, then re-run this cell."
            )
        print(f"Updating existing checkout at {CHECKOUT_DIR} ...")
        _run(["-C", str(CHECKOUT_DIR), "fetch", "origin"], auth=True)
        _run(["-C", str(CHECKOUT_DIR), "checkout", BRANCH])
        _run(["-C", str(CHECKOUT_DIR), "merge", "--ff-only", f"origin/{BRANCH}"])

    final_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
    if "@" in final_url or GITHUB_TOKEN in final_url:
        raise RuntimeError("origin URL unexpectedly contains credentials; aborting.")

    os.chdir(CHECKOUT_DIR)
    if str(CHECKOUT_DIR) not in sys.path:
        sys.path.insert(0, str(CHECKOUT_DIR))

    print("Installing the 'gemma' extra ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[gemma]"],
        check=True,
    )

    branch_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "--abbrev-ref", "HEAD"])
    sha_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "HEAD"])
    print(f"checked out: {branch_now} @ {sha_now}")
    assert (CHECKOUT_DIR / "jlens").is_dir(), f"{CHECKOUT_DIR}/jlens not found"


In [ ]:
# 1. Execution gates. Model loading and CUDA are opt-in; outside Colab the
# defaults keep everything in the light (no-download) path so the notebook's
# structure can be validated locally.
import os
import sys

IN_COLAB = "google.colab" in sys.modules
os.environ.setdefault("JLENS_ALLOW_GEMMA", "1" if IN_COLAB else "0")
os.environ.setdefault("JLENS_DEVICE_MAP", "cuda" if IN_COLAB else "")
# Resuming a specific prior run is an explicit opt-in (its exact RUN_DIR):
# os.environ["CAUSAL_RESUME_RUN_DIR"] = ".../runs/causal_..."
ALLOW_MODEL_LOAD = os.environ.get("JLENS_ALLOW_GEMMA", "0") == "1"
DEVICE_MAP = os.environ.get("JLENS_DEVICE_MAP") or None
print(f"IN_COLAB={IN_COLAB}  ALLOW_MODEL_LOAD={ALLOW_MODEL_LOAD}  DEVICE_MAP={DEVICE_MAP}")


In [ ]:
# 2. Environment and provenance (no model load).
import json
import pathlib

import torch

from jlens.metadata import environment_manifest

ENV = environment_manifest()
print(json.dumps(ENV, indent=2))


## Persisting outputs to Google Drive

Each execution creates a fresh timestamped run directory under
`MyDrive/jacobian-lens-gemma/runs/` (never overwriting a prior run) and
checkpoints after **every completed condition**, so an interrupted run
loses at most one condition and resumes append-safely.

In [ ]:
# 3. Google Drive persistence (Colab only). No-op outside Colab.
from pathlib import Path

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError(
            f"failed to mount Google Drive at /content/drive: {exc}. Approve "
            "the Drive authorization prompt when it appears, then re-run this cell."
        ) from exc

    DRIVE_MOUNT = Path("/content/drive")
    if not DRIVE_MOUNT.is_dir():
        raise RuntimeError("Drive did not mount successfully.")

    PERSIST_ROOT = DRIVE_MOUNT / "MyDrive" / "jacobian-lens-gemma"
    RUNS_ROOT = PERSIST_ROOT / "runs"
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"RUNS_ROOT = {RUNS_ROOT}")
else:
    PERSIST_ROOT = None
    RUNS_ROOT = Path("runs")  # local checkout: read the archived runs
    print("Not in Colab — using the local runs/ directory (read) and "
          "artifacts/causal_smoke (write).")


In [ ]:
# 4. Load and validate the causal configuration; create (or resume,
# explicitly and fingerprint-validated) the run directory. Completed runs
# (run_metadata.json present) refuse resumption.
from datetime import datetime, timezone

from jlens.interventions import assert_run_resumable
from jlens.metadata import config_fingerprint, execution_record, load_causal_config, write_metadata

CONFIG_PATH = "configs/gemma_jspace_causal_smoke.yaml"
CONFIG = load_causal_config(CONFIG_PATH)
FINGERPRINT = config_fingerprint(CONFIG)
INT = CONFIG["intervention"]
print(f"config: {CONFIG_PATH}\nfingerprint: {FINGERPRINT}")
print(f"layers={INT['layers']}  multipliers={INT['multipliers']}  families={INT['families']}")

EXECUTION = execution_record(
    configured_allow_model_load=CONFIG["model"]["allow_model_load"],
    resolved_allow_model_load=ALLOW_MODEL_LOAD,
    model_loaded=False,  # updated after the actual load
    override_source="notebook:JLENS_ALLOW_GEMMA" if ALLOW_MODEL_LOAD else None,
)

RESUME_RUN_DIR_ENV = os.environ.get("CAUSAL_RESUME_RUN_DIR") or None
if IN_COLAB:
    if RESUME_RUN_DIR_ENV is not None:
        RUN_DIR = pathlib.Path(RESUME_RUN_DIR_ENV)
        assert_run_resumable(str(RUN_DIR))  # refuses completed runs
        started_path = RUN_DIR / "run_started.json"
        if not started_path.is_file():
            raise RuntimeError(
                f"CAUSAL_RESUME_RUN_DIR={RUN_DIR} has no run_started.json; "
                "point it at a previously started causal run directory"
            )
        prior = json.load(open(started_path, encoding="utf-8"))
        if prior.get("config_fingerprint") != FINGERPRINT:
            raise RuntimeError(
                f"refusing to resume {RUN_DIR}: recorded fingerprint "
                f"{prior.get('config_fingerprint')!r} != active {FINGERPRINT!r}"
            )
        RUN_ID = RUN_DIR.name
        print(f"resuming explicit run: {RUN_DIR} (fingerprint verified)")
    else:
        RUN_ID = (f"causal_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')}_"
                  f"{FINGERPRINT.removeprefix('sha256:')[:12]}")
        RUN_DIR = RUNS_ROOT / RUN_ID
        RUN_DIR.mkdir(parents=True, exist_ok=False)  # fresh-run overwrite protection
    OUTPUT_DIR = RUN_DIR / "artifacts"
else:
    RUN_ID = None
    RUN_DIR = None
    OUTPUT_DIR = pathlib.Path(CONFIG["paths"]["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_DIR is not None and not (RUN_DIR / "run_started.json").is_file():
    write_metadata(str(RUN_DIR / "run_started.json"), {
        "run_id": RUN_ID,
        "mode": "causal_smoke",
        "config_fingerprint": FINGERPRINT,
        "execution": EXECUTION,
    })
if RUN_DIR is not None:
    write_metadata(str(RUN_DIR / "resolved_config.json"), {
        "config": CONFIG,
        "config_fingerprint": FINGERPRINT,
        "execution": EXECUTION,
    })
print(f"RUN_DIR = {RUN_DIR}\nOUTPUT_DIR = {OUTPUT_DIR}")


In [ ]:
# 5. Lightweight validation suite (CPU, mocks, no network) — must pass
# before any real-model work, exactly as in the prior notebooks.
import subprocess

proc = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--no-header"],
    capture_output=True, text=True,
)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise RuntimeError("local test suite failed; fix before running the model")


In [ ]:
# 6. Locate and verify the FROZEN pilot lens and the completed jspace run.
# Read-only: fingerprints, revisions, layers. Never refitted, never rerun.
from jlens.lens import JacobianLens
from jlens.metadata import file_sha256

LENS_CFG = CONFIG["lens"]
PILOT_RUN_DIR = RUNS_ROOT / LENS_CFG["run_dir_name"]
LENS_PATH = PILOT_RUN_DIR / LENS_CFG["artifact_relpath"]
if not LENS_PATH.is_file():
    raise RuntimeError(
        f"pilot lens not found at {LENS_PATH}. In Colab, RUNS_ROOT must be the "
        "Drive runs/ directory containing the completed pilot run "
        f"{LENS_CFG['run_dir_name']!r}."
    )

lens_sha = file_sha256(str(LENS_PATH))
print(f"lens file: {LENS_PATH}\nsha256:    {lens_sha}")
if LENS_CFG["expect_file_sha256"] and lens_sha != LENS_CFG["expect_file_sha256"]:
    raise RuntimeError(
        f"lens fingerprint mismatch: expected {LENS_CFG['expect_file_sha256']}, "
        f"got {lens_sha} — refusing to intervene against an unexpected lens"
    )

LENS = JacobianLens.load(str(LENS_PATH))
assert LENS.source_layers == LENS_CFG["expect_source_layers"], LENS.source_layers
assert LENS.n_prompts == LENS_CFG["expect_n_prompts"], LENS.n_prompts
assert LENS.d_model == CONFIG["model"]["expect_d_model"], LENS.d_model
for layer, J in LENS.jacobians.items():
    assert J.shape == (LENS.d_model, LENS.d_model), (layer, J.shape)
    assert torch.isfinite(J).all(), f"non-finite J at layer {layer}"

pilot_meta_path = PILOT_RUN_DIR / "run_metadata.json"
PILOT_META = json.load(open(pilot_meta_path, encoding="utf-8"))
assert PILOT_META["model_revision"] == LENS_CFG["expect_model_revision"], (
    PILOT_META["model_revision"]
)
LENS_VERIFICATION = {
    "lens_path": str(LENS_PATH),
    "file_sha256": lens_sha,
    "source_layers": LENS.source_layers,
    "n_prompts": LENS.n_prompts,
    "d_model": LENS.d_model,
    "all_finite": True,
    "pilot_run_id": PILOT_META["run_id"],
    "pilot_model_revision": PILOT_META["model_revision"],
    "pilot_config_fingerprint": PILOT_META["config_fingerprint"],
}

# Source jspace run: verify fingerprint and load the k=10 cone records for
# the manifest examples at the steering layers.
SRC_CFG = CONFIG["source"]
JSPACE_RUN_DIR = RUNS_ROOT / SRC_CFG["jspace_run_dir_name"]
JSPACE_META_PATH = JSPACE_RUN_DIR / "run_metadata.json"
if not JSPACE_META_PATH.is_file():
    raise RuntimeError(
        f"completed jspace run not found at {JSPACE_RUN_DIR}; the causal smoke "
        "run consumes its cone records and cannot proceed without them"
    )
JSPACE_META = json.load(open(JSPACE_META_PATH, encoding="utf-8"))
if (SRC_CFG["expect_config_fingerprint"]
        and JSPACE_META["config_fingerprint"] != SRC_CFG["expect_config_fingerprint"]):
    raise RuntimeError(
        f"jspace run fingerprint mismatch: expected "
        f"{SRC_CFG['expect_config_fingerprint']}, got {JSPACE_META['config_fingerprint']}"
    )
assert JSPACE_META["lens_verification"]["file_sha256"] == lens_sha, (
    "jspace run used a different lens than the one verified above")

from jlens.cones import load_cone_records

K = SRC_CFG["cones_k"]
CONE_FILES = {}
CONES_BY_KEY = {}
for layer in INT["layers"]:
    rel = f"artifacts/cones/cones_layer{layer}_k{K}.json"
    path = JSPACE_RUN_DIR / rel
    records = load_cone_records(str(path))
    CONE_FILES[rel] = file_sha256(str(path))
    for record in records:
        CONES_BY_KEY[(record["prompt_hash"], record["position"], record["layer"])] = record
print(f"loaded {len(CONES_BY_KEY)} cone records from {len(CONE_FILES)} files")
print(json.dumps(LENS_VERIFICATION, indent=2))


In [ ]:
# 7. Select manifest examples and build the deterministic condition plan.
# Pure record logic — runs in the light path too (no model needed). Every
# condition gets a deterministic ID; the plan is saved before execution.
from jlens.explorer_export import example_id as make_example_id
from jlens.interventions import TARGET_KINDS, condition_id

MANIFEST = json.load(open(CONFIG["examples"]["manifest_path"], encoding="utf-8"))
CAPTURE_BY_KEY = {
    (m["slug"], m["position"]): m for m in JSPACE_META["capture_meta"]
}

SELECTED = []
for entry in MANIFEST["examples"]:
    slug, prompt_hash = entry["slug"], entry["prompt_hash"]
    for position in entry["positions"]:
        capture = CAPTURE_BY_KEY.get((slug, position))
        if capture is None:
            raise RuntimeError(f"manifest example {slug}@{position} not in jspace capture_meta")
        if capture["prompt_hash"] != prompt_hash:
            raise RuntimeError(
                f"{slug}: manifest prompt_hash {prompt_hash} != recorded "
                f"{capture['prompt_hash']} — manifest and run disagree")
        for layer in INT["layers"]:
            if (prompt_hash, position, layer) not in CONES_BY_KEY:
                raise RuntimeError(f"no k={K} cone for {slug}@{position} layer {layer}")
        SELECTED.append({
            "slug": slug,
            "prompt_hash": prompt_hash,
            "format": entry["format"],
            "position": position,
            "example_id": make_example_id("text", slug, prompt_hash),
            "selection_reason": entry["selection_reason"],
            "model_top1_id": capture["model_top1_id"],
            "model_top1_token": capture["model_top1_token"],
            "seq_len": capture["seq_len"],
        })
print(f"selected {len(SELECTED)} (example, position) targets")

def _atom_pick(cone, kind, model_top1_id):
    """Mirror of jlens.interventions.cone_delta's selection rule, for the
    plan only (cone_delta remains authoritative at execution time)."""
    ids = [int(t) for t in cone["effective_token_ids"]]
    coeffs = [float(c) for c in cone["effective_coefficients"]]
    labels = list(cone["effective_labels"])
    if kind == "output_atom_contribution":
        if model_top1_id not in ids:
            return None
        i = ids.index(model_top1_id)
        return ids[i], labels[i]
    if kind == "top_non_output_atom_contribution":
        cands = [(c, -i) for i, (t, c) in enumerate(zip(ids, coeffs)) if t != model_top1_id]
        if not cands:
            return None
        i = -max(cands)[1]
        return ids[i], labels[i]
    return None, None  # full cone

PLAN = []
for target in SELECTED:
    for layer in INT["layers"]:
        cone = CONES_BY_KEY[(target["prompt_hash"], target["position"], layer)]
        for family in INT["families"]:
            assert family in TARGET_KINDS, family
            pick = _atom_pick(cone, family, target["model_top1_id"])
            if pick is None:
                print(f"skip {family} for {target['slug']}@L{layer}: no eligible atom")
                continue
            atom_id, atom_label = pick
            for multiplier in INT["multipliers"]:
                cid = condition_id(
                    example_id=target["example_id"], layer=layer,
                    position=target["position"], target_kind=family,
                    multiplier=float(multiplier), atom_token_id=atom_id,
                    norm_preserving=INT["norm_preserving"],
                )
                PLAN.append({
                    "condition_id": cid, "example_id": target["example_id"],
                    "slug": target["slug"], "layer": layer,
                    "position": target["position"], "target_kind": family,
                    "atom_token_id": atom_id, "atom_label": atom_label,
                    "multiplier": float(multiplier),
                    "norm_preserving": INT["norm_preserving"],
                    "control_family": None, "matched_target_condition_id": None,
                })
                if multiplier != 0.0:
                    for control in INT["controls"]:
                        ctrl_id = condition_id(
                            example_id=target["example_id"], layer=layer,
                            position=target["position"], target_kind=control,
                            multiplier=float(multiplier), atom_token_id=atom_id,
                            norm_preserving=INT["norm_preserving"],
                            variant=f"matched:{cid}",
                        )
                        PLAN.append({
                            "condition_id": ctrl_id, "example_id": target["example_id"],
                            "slug": target["slug"], "layer": layer,
                            "position": target["position"], "target_kind": control,
                            "atom_token_id": None, "atom_label": None,
                            "multiplier": float(multiplier),
                            "norm_preserving": INT["norm_preserving"],
                            "control_family": control,
                            "matched_target_condition_id": cid,
                        })

assert len({p["condition_id"] for p in PLAN}) == len(PLAN), "condition ID collision"
print(f"condition plan: {len(PLAN)} conditions "
      f"({sum(1 for p in PLAN if not p['control_family'])} targeted, "
      f"{sum(1 for p in PLAN if p['control_family'])} controls)")

if RUN_DIR is not None:
    write_metadata(str(RUN_DIR / "selected_examples.json"), {
        "manifest_path": CONFIG["examples"]["manifest_path"],
        "manifest_version": MANIFEST.get("version"),
        "selected": SELECTED,
    })
    write_metadata(str(RUN_DIR / "condition_plan.json"), {
        "n_conditions": len(PLAN),
        "plan": PLAN,
    })


In [ ]:
# 8. Load the SAME immutable Gemma revision the lens was fitted on (gated).
from jlens.gemma4 import load_gemma4, verify_architecture

MODEL = None
LOAD_INFO = None
ARCH_REPORT = None
if not ALLOW_MODEL_LOAD:
    print("JLENS_ALLOW_GEMMA != 1 — light path: skipping model load. The "
          "condition plan above is still valid and was saved.")
else:
    MODEL, LOAD_INFO = load_gemma4(
        CONFIG["model"]["repo_id"],
        revision=CONFIG["model"]["revision"],
        dtype=getattr(torch, CONFIG["model"]["dtype"]),
        device_map=DEVICE_MAP,
        allow_model_load=True,
    )
    for parameter in MODEL._hf_model.parameters():
        parameter.requires_grad_(False)
    ARCH_REPORT = verify_architecture(
        MODEL,
        expect_n_layers=CONFIG["model"]["expect_n_layers"],
        expect_d_model=CONFIG["model"]["expect_d_model"],
        expect_vocab_size=CONFIG["model"]["expect_vocab_size"],
    ).to_dict()
    assert LOAD_INFO["model_revision"] == LENS_VERIFICATION["pilot_model_revision"], (
        "model revision differs from the lens's fit revision")
    EXECUTION = execution_record(
        configured_allow_model_load=CONFIG["model"]["allow_model_load"],
        resolved_allow_model_load=True,
        model_loaded=True,
        override_source="notebook:JLENS_ALLOW_GEMMA",
    )
    print(f"loaded {LOAD_INFO['model_repo_id']} @ {LOAD_INFO['model_revision']}")


In [ ]:
# 9. Render prompts exactly as the jspace run did and verify the hashes,
# then run the BASELINE-PARITY GATE: (a) unhooked forward, (b) multiplier-0
# hook, (c) identical-copy writeback. Recorded; aborts on tolerance failure.
import hashlib

from jlens.evaluation import load_eval_prompts_v2
from jlens.interventions import parity_report, residual_intervention

BASELINES = {}
PARITY = None
if MODEL is None:
    print("light path — skipping parity (requires the model).")
else:
    EVAL_PROMPTS_PATH = "configs/prompts/eval_prompts_v2.json"
    rows = load_eval_prompts_v2(EVAL_PROMPTS_PATH, MODEL.tokenizer)
    TEXT_BY_SLUG = {row["slug"]: row["text"] for row in rows}
    for target in SELECTED:
        text = TEXT_BY_SLUG.get(target["slug"])
        if text is None:
            raise RuntimeError(f"prompt for {target['slug']} not found in eval_prompts_v2")
        rendered_hash = hashlib.sha256(text.encode()).hexdigest()[:16]
        if rendered_hash != target["prompt_hash"]:
            raise RuntimeError(
                f"{target['slug']}: rendered prompt hash {rendered_hash} != "
                f"recorded {target['prompt_hash']} — tokenizer/template drift; aborting")
        target["text"] = text

    @torch.no_grad()
    def final_logits(input_ids):
        """Softcapped final logits at the last position — the model's actual
        sampling pathway (rankings match the pre-softcap readout exactly)."""
        out = MODEL.forward(input_ids)
        logits = out.logits if hasattr(out, "logits") else out[0]
        return logits[0, -1].detach().float().cpu()

    blocks = MODEL.layers
    tol = float(CONFIG["parity"]["max_abs_logit_diff_tol"])
    checks = []
    for target in SELECTED:
        input_ids = MODEL.encode(target["text"], max_length=512)
        base = final_logits(input_ids)
        BASELINES[(target["example_id"], target["position"])] = base
        for layer in INT["layers"]:
            cone = CONES_BY_KEY[(target["prompt_hash"], target["position"], layer)]
            probe = torch.ones(LENS.d_model)  # any finite delta; multiplier 0
            with residual_intervention(blocks, layer, position=target["position"],
                                       delta=probe, multiplier=0.0) as stats:
                mult0 = final_logits(input_ids)
            with residual_intervention(blocks, layer, position=target["position"],
                                       delta=None) as stats:
                writeback = final_logits(input_ids)
            for name, other in (("multiplier_zero", mult0), ("writeback", writeback)):
                report = parity_report(base, other)
                report.update({"slug": target["slug"], "layer": layer, "check": name})
                checks.append(report)
                if report["max_abs_logit_diff"] > tol or not report["top1_identical"]:
                    raise RuntimeError(
                        f"BASELINE PARITY FAILED ({target['slug']} L{layer} {name}): "
                        f"{report} exceeds tol={tol}; aborting before any intervention")
    PARITY = {
        "tolerance_max_abs_logit_diff": tol,
        "readout": "softcapped final logits (model sampling pathway)",
        "n_checks": len(checks),
        "worst_max_abs_logit_diff": max(c["max_abs_logit_diff"] for c in checks),
        "all_top1_identical": all(c["top1_identical"] for c in checks),
        "min_top10_overlap": min(c["top10_overlap"] for c in checks),
        "checks": checks,
    }
    write_metadata(str(OUTPUT_DIR / "baseline_parity.json"), PARITY)
    print(json.dumps({k: v for k, v in PARITY.items() if k != "checks"}, indent=2))


In [ ]:
# 10. Execute the condition plan with per-condition checkpointing.
# Deltas come from the recorded cones via the audited pursuit convention
# (raw atoms, nonnegative coefficients); controls are deterministic and
# exactly norm-matched. Completed condition IDs are skipped on resume.
from jlens.interventions import (
    _derived_seed,
    DeltaInfo,
    append_record,
    completed_condition_ids,
    cone_delta,
    isotropic_random_direction,
    logit_metrics,
    make_intervention_record,
)
from jlens.pursuit import JSpaceDictionary

RECORDS_PATH = str(OUTPUT_DIR / "intervention_records.jsonl")

if MODEL is None:
    print("light path — skipping execution.")
else:
    done = completed_condition_ids(RECORDS_PATH)
    print(f"{len(done)} conditions already recorded; executing the rest")

    @torch.no_grad()
    def greedy_completion(input_ids, *, hook=None, max_new_tokens):
        """Greedy decode WITHOUT KV cache: each step reruns the full forward
        so the (absolute-position) intervention applies identically at every
        step. Slow but exact; fine at smoke scale."""
        ids = input_ids.clone()
        for _ in range(max_new_tokens):
            if hook is None:
                logits = final_logits(ids)
            else:
                with residual_intervention(MODEL.layers, hook["layer"],
                                           position=hook["position"],
                                           delta=hook["delta"],
                                           multiplier=hook["multiplier"],
                                           norm_preserving=hook["norm_preserving"]):
                    logits = final_logits(ids)
            next_id = int(logits.argmax())
            ids = torch.cat([ids, torch.tensor([[next_id]], device=ids.device)], dim=1)
        return MODEL.tokenizer.decode(ids[0, input_ids.shape[1]:].tolist())

    def decode_token(token_id):
        return MODEL.tokenizer.decode([token_id])

    TARGETS_BY_ID = {t["example_id"]: t for t in SELECTED}
    completions_base = {}
    gen = INT["generate_completions"]
    max_new = INT["completion_max_new_tokens"]

    dictionaries = {}
    for layer in INT["layers"]:
        dictionaries[layer] = JSpaceDictionary.from_lens(
            LENS, layer, MODEL._lm_head.weight,
            device=MODEL._lm_head.weight.device,
            build_chunk_rows=16384,
        )
    print(f"built dictionaries for layers {sorted(dictionaries)}")

    # Cache targeted delta norms so each matched control can reproduce them
    # even when its targeted condition was completed in an earlier session.
    def targeted_delta(plan_row):
        cone = CONES_BY_KEY[(TARGETS_BY_ID[plan_row["example_id"]]["prompt_hash"],
                             plan_row["position"], plan_row["layer"])]
        model_top1 = TARGETS_BY_ID[plan_row["example_id"]]["model_top1_id"]
        return cone_delta(dictionaries[plan_row["layer"]].atoms, cone,
                          target_kind=plan_row["target_kind"], model_top1_id=model_top1)

    PLAN_BY_ID = {p["condition_id"]: p for p in PLAN}
    control_matching = []
    for index, plan_row in enumerate(PLAN):
        cid = plan_row["condition_id"]
        if cid in done:
            continue
        target = TARGETS_BY_ID[plan_row["example_id"]]
        input_ids = MODEL.encode(target["text"], max_length=512)
        resolved_pos = input_ids.shape[1] + plan_row["position"]
        base = BASELINES[(plan_row["example_id"], plan_row["position"])]

        if plan_row["control_family"] is None:
            delta, info = targeted_delta(plan_row)
        else:
            matched = PLAN_BY_ID[plan_row["matched_target_condition_id"]]
            _, matched_info = targeted_delta(matched)
            delta, info = isotropic_random_direction(
                LENS.d_model,
                match_norm=matched_info.delta_norm,
                seed=_derived_seed(cid),
                device=dictionaries[plan_row["layer"]].device,
                matched_target_condition_id=matched["condition_id"],
            )
            control_matching.append({
                "control_condition_id": cid,
                "matched_target_condition_id": matched["condition_id"],
                "matched_delta_norm": matched_info.delta_norm,
                "control_delta_norm": info.delta_norm,
                "seed": info.seed,
            })

        hook = {"layer": plan_row["layer"], "position": resolved_pos,
                "delta": delta, "multiplier": plan_row["multiplier"],
                "norm_preserving": plan_row["norm_preserving"]}
        with residual_intervention(MODEL.layers, hook["layer"], position=hook["position"],
                                   delta=delta, multiplier=hook["multiplier"],
                                   norm_preserving=hook["norm_preserving"]) as stats:
            after = final_logits(input_ids)

        metrics = logit_metrics(base, after,
                                target_token_id=target["model_top1_id"],
                                top_k=CONFIG["eval"]["top_k"], decode=decode_token)
        completion_before = completion_after = None
        if gen and plan_row["multiplier"] != 0.0:
            key = plan_row["example_id"]
            if key not in completions_base:
                completions_base[key] = greedy_completion(input_ids, max_new_tokens=max_new)
            completion_before = completions_base[key]
            completion_after = greedy_completion(input_ids, hook=hook, max_new_tokens=max_new)

        record = make_intervention_record(
            condition=cid,
            example_id=plan_row["example_id"],
            layer=plan_row["layer"],
            position=plan_row["position"],
            delta_info=info,
            multiplier=plan_row["multiplier"],
            stats=stats,
            metrics=metrics,
            status="measured",
            control_family=plan_row["control_family"],
            completion_before=completion_before,
            completion_after=completion_after,
            provenance={
                "run_id": RUN_ID,
                "jspace_run_id": JSPACE_META["run_id"],
                "lens_fingerprint": LENS_VERIFICATION["file_sha256"],
                "model_revision": LOAD_INFO["model_revision"],
                "config_fingerprint": FINGERPRINT,
                "resolved_position": resolved_pos,
                "target_token_source": "model top-1 at the intervened position (jspace capture_meta)",
                "readout": "softcapped final logits (model sampling pathway)",
            },
        )
        append_record(RECORDS_PATH, record)  # checkpoint: one condition, one line
        if (index + 1) % 10 == 0:
            print(f"  {index + 1}/{len(PLAN)} conditions done")
        del delta
    if control_matching and RUN_DIR is not None:
        write_metadata(str(OUTPUT_DIR / "control_matching.json"),
                       {"n_controls": len(control_matching), "pairs": control_matching})
    del dictionaries
    print("execution complete")


In [ ]:
# 11. Export: CSV, explorer causal bundle (schema-validated when jsonschema
# is available), and a small analysis summary.
import csv
from datetime import datetime, timezone

from jlens.explorer_export import (
    assemble_bundle,
    intervention_to_causal_record,
    make_provenance,
    write_bundle,
)
from jlens.interventions import load_records

if MODEL is None:
    print("light path — skipping export.")
else:
    records = load_records(RECORDS_PATH)
    print(f"{len(records)} intervention records")

    # CSV (lists JSON-encoded so the table stays flat).
    csv_path = str(OUTPUT_DIR / "intervention_records.csv")
    field_names = sorted({key for record in records for key in record})
    with open(csv_path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=field_names)
        writer.writeheader()
        for record in records:
            writer.writerow({
                key: json.dumps(value, ensure_ascii=False)
                if isinstance(value, (dict, list)) else value
                for key, value in record.items()
            })

    provenance = make_provenance(
        source_run_ids=[RUN_ID, JSPACE_META["run_id"]],
        model_repo_id=LOAD_INFO["model_repo_id"],
        model_revision=LOAD_INFO["model_revision"],
        created_utc=datetime.now(timezone.utc).isoformat(timespec="seconds"),
        data_status="measured",
        modalities_present=["text"],
        lens_fingerprint=LENS_VERIFICATION["file_sha256"],
        source_artifact_fingerprints=dict(CONE_FILES),
        implementation_commit=ENV.get("local_commit"),
        notes="Measured causal steering smoke run; see docs/causal_smoke_run.md.",
    )
    bundle = assemble_bundle(
        provenance=provenance,
        causal_records=[intervention_to_causal_record(r) for r in records],
        causal_baseline_parity={k: v for k, v in PARITY.items() if k != "checks"},
    )
    schema_path = "schemas/explorer_bundle.schema.json"
    try:
        import jsonschema  # noqa: F401
        bundle_sha = write_bundle(bundle, str(OUTPUT_DIR / "explorer_causal_bundle.json"),
                                  schema_path=schema_path)
    except ImportError:
        print("jsonschema not installed — writing without schema validation")
        bundle_sha = write_bundle(bundle, str(OUTPUT_DIR / "explorer_causal_bundle.json"))
    print(f"explorer bundle: {bundle_sha}")

    # Descriptive summary: targeted vs matched control per family/multiplier.
    def mean(values):
        values = [v for v in values if v is not None]
        return sum(values) / len(values) if values else None

    summary_rows = []
    for family in INT["families"]:
        for multiplier in INT["multipliers"]:
            targeted = [r for r in records
                        if r["target_kind"] == family and r["multiplier"] == multiplier
                        and not r["control_family"]]
            controls = [r for r in records
                        if r["control_family"] and r["multiplier"] == multiplier
                        and PLAN_BY_ID.get(r["matched_target_condition_id"], {}).get("target_kind") == family]
            if not targeted:
                continue
            summary_rows.append({
                "family": family, "multiplier": multiplier, "n": len(targeted),
                "mean_target_logit_delta": mean([r["target_logit_delta"] for r in targeted]),
                "mean_kl": mean([r["kl_divergence_after_vs_before"] for r in targeted]),
                "top1_change_rate": mean([
                    0.0 if r["top1_before"]["token_id"] == r["top1_after"]["token_id"] else 1.0
                    for r in targeted]),
                "control_mean_target_logit_delta": mean([r["target_logit_delta"] for r in controls]),
                "control_mean_kl": mean([r["kl_divergence_after_vs_before"] for r in controls]),
            })
    ANALYSIS = {
        "schema": "jlens.causal_smoke.analysis.v1",
        "n_records": len(records),
        "rows": summary_rows,
        "notes": (
            "Descriptive means over a 4-example smoke set; not a benchmark. "
            "Causal claims are limited to these measured conditions."),
    }
    write_metadata(str(OUTPUT_DIR / "analysis_summary.json"), ANALYSIS)
    print(json.dumps(summary_rows, indent=2)[:2000])


In [ ]:
# 12. Run manifest + human-readable summary into the run directory.
if MODEL is None:
    print("model not loaded — nothing to summarize (light path).")
else:
    manifest = {
        "run_id": RUN_ID,
        "run_dir": str(RUN_DIR),
        "mode": "causal_smoke",
        "config": CONFIG,
        "config_fingerprint": FINGERPRINT,
        "execution": EXECUTION,
        "lens_verification": LENS_VERIFICATION,
        "jspace_run_id": JSPACE_META["run_id"],
        "jspace_config_fingerprint": JSPACE_META["config_fingerprint"],
        "source_cone_fingerprints": CONE_FILES,
        "load_info": LOAD_INFO,
        "architecture_report": ARCH_REPORT,
        "n_conditions_planned": len(PLAN),
        "n_conditions_recorded": len(load_records(RECORDS_PATH)),
        "baseline_parity": {k: v for k, v in PARITY.items() if k != "checks"},
        "environment": ENV,
        "notes": (
            "Frozen pilot lens; cones consumed read-only from the completed "
            "jspace run. h' = h + multiplier * delta at block_output; "
            "controls are deterministic norm-matched random directions. "
            "Effects are measured only at the recorded multipliers."),
    }
    write_metadata(str(RUN_DIR / "run_metadata.json"), manifest)

    lines = [
        f"# Run {RUN_ID}",
        "",
        "- mode: causal_smoke (measured interventions on the frozen pilot lens's J-space cones)",
        f"- lens: {LENS_VERIFICATION['file_sha256']}",
        f"- model: {LOAD_INFO['model_repo_id']} @ {LOAD_INFO['model_revision']}",
        f"- layers: {INT['layers']}; multipliers: {INT['multipliers']}; families: {INT['families']}",
        f"- conditions recorded: {manifest['n_conditions_recorded']} / {len(PLAN)} planned",
        f"- parity: worst max-abs logit diff {PARITY['worst_max_abs_logit_diff']:.5f} "
        f"(tol {PARITY['tolerance_max_abs_logit_diff']})",
        "- artifacts: baseline_parity.json, intervention_records.jsonl/.csv, "
        "control_matching.json, explorer_causal_bundle.json, analysis_summary.json",
        "",
        "Next: copy explorer_causal_bundle.json to explorer/public/data/measured/causal.json "
        "(see docs/causal_smoke_run.md).",
    ]
    with open(RUN_DIR / "summary.md", "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))
    print("\n".join(lines))


## Summary and limitations

- **What this run establishes when it completes:** measured before/after
  next-token distributions for adding/removing recorded cone components at
  layers 35/38 on four examples, against exactly norm-matched random
  controls — nothing more. A logit shift on four prompts is a smoke result,
  not a benchmark.
- **What it does not establish:** that cone atoms are "concepts", that
  effects generalize beyond these prompts, or anything about layers not
  measured. Multipliers are measured points; nothing between them is
  interpolated.
- The multiplier-0 and writeback parity checks bound hook-induced numeric
  drift *before* any intervention is trusted; the tolerance and the worst
  observed difference are both recorded in `baseline_parity.json`.
- After the run: copy `artifacts/explorer_causal_bundle.json` into
  `explorer/public/data/measured/causal.json` — the explorer then shows
  measured records and stops offering the synthetic causal fixture.